In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/


In [ ]:
!kaggle datasets download -d smaranjitghose/corn-or-maize-leaf-disease-dataset

 95% 153M/161M [00:01<00:00, 118MB/s]
100% 161M/161M [00:01<00:00, 113MB/s]


In [ ]:
import zipfile
zip_ref=zipfile.ZipFile('/content/corn-or-maize-leaf-disease-dataset.zip','r')
zip_ref.extractall('/content')
zip_ref.close()


In [55]:
import cv2
import os
import random
import imghdr
from keras.preprocessing.image import ImageDataGenerator
def augmentation(dir_path,save_path,num_augmented_img):
    datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        vertical_flip=True,
        brightness_range=[0.5, 1.5],
        channel_shift_range=20,
        fill_mode='nearest'
    )

    savedir_path = save_path
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    image_directory = dir_path
    image_files = [os.path.join(image_directory, file) for file in os.listdir(image_directory)]
    print(image_files)
    for i in range(num_augmented_img):
        img_path = random.choice(image_files)
        img = cv2.imread(img_path)
        if img is None:
            print("Error: Unable to load image", img_path)
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.reshape((1,) + img.shape)
        augmented_image = next(datagen.flow(img, batch_size=1))[0].astype('uint8')

        cv2.imwrite(os.path.join(save_path, f"augmented_{i}.jpg"), cv2.cvtColor(augmented_image, cv2.COLOR_RGB2BGR))
def delete_images_with_prefix(directory, prefix):
    all_files = os.listdir(directory)
    files_to_delete = [file for file in all_files if file.startswith(prefix)]
    for file_to_delete in files_to_delete:
        file_path = os.path.join(directory, file_to_delete)
        # Verify if the file is an image file
        if imghdr.what(file_path) is not None:
            os.remove(file_path)
            print(f"Deleted: {file_path}")

def print_augmented_images(directory):
    """
    Print the filenames of all images in the directory that start with 'augmented'.

    Parameters:
    directory (str): Path to the directory containing images.
    """
    # Get all files in the directory
    files = os.listdir(directory)

    # Filter files that start with 'augmented'
    augmented_files = [file for file in files if file.startswith('augmented')]

    print(f'the number of images in dir {directory} :{len(augmented_files)}')



In [ ]:
path1="/content/data/Blight"
path2="/content/data/Common_Rust"
path3="/content/data/Gray_Leaf_Spot"
path4="/content/data/Healthy"
num=300
save_path1="/content/data/Blight"
save_path2="/content/data/Common_Rust"
save_path3="/content/data/Gray_Leaf_Spot"
save_path4="/content/data/Healthy"

# save_paths=[save_path1,save_path2,save_path3,save_path4]
# paths=[path1,path2,path3,path4]
# for index,path in enumerate(paths):
#   augmentation(path,save_paths[index],num)
# print("DATA AUGMENTATION COMPLETED")

prefix1='augmented'
for path in paths:
    delete_images_with_prefix(path, prefix1)
    # delete_images_with_prefix(path, prefix2)
print("NOW the data is cleaned")




In [59]:
print_augmented_images(path1)

the number of images in dir /content/data/Blight :0


In [43]:
import shutil
from sklearn.model_selection import train_test_split

def splitting_Data(dataset_root,root):

    classes = os.listdir(dataset_root)
    print(classes)
    test_size = 0.2
    train_root = os.path.join(root, "train")
    test_root = os.path.join(root, "test")
    if not os.path.exists(train_root):
        os.makedirs(train_root)
    if not os.path.exists(test_root):
        os.makedirs(test_root)
    for class_name in classes:
        class_path = os.path.join(dataset_root, class_name)
        image_files = [file for file in os.listdir(class_path) if cv2.imread(os.path.join(class_path, file)) is not None]
        train_files, test_files = train_test_split(image_files, test_size=test_size, random_state=42)
        train_class_dir = os.path.join(train_root, class_name)
        test_class_dir = os.path.join(test_root, class_name)
        if not os.path.exists(train_class_dir):
            os.makedirs(train_class_dir)
        if not os.path.exists(test_class_dir):
            os.makedirs(test_class_dir)
        for file in train_files:
            shutil.copy(os.path.join(class_path, file), os.path.join(train_class_dir, file))
        for file in test_files:
            shutil.copy(os.path.join(class_path, file), os.path.join(test_class_dir, file))
    print(f"Data Had been successfully Splited into Train and Test :{root} ")


In [44]:
dir_root="/content/data"
save_root='/content/new data'
splitting_Data(dir_root,save_root)

['Healthy', 'Common_Rust', 'Gray_Leaf_Spot', 'Blight']
Data Had been successfully Splited into Train and Test :/content/new data 


In [60]:
from keras.preprocessing.image import ImageDataGenerator
def Generators(train_dir,validation_dir):
    train_datagen = ImageDataGenerator(rescale=1./255)
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(256, 256),
        batch_size=32,
        class_mode='categorical'
    )

    validation_datagen = ImageDataGenerator(rescale=1./255)
    validation_generator = validation_datagen.flow_from_directory(
        validation_dir,
        target_size=(256, 256),
        batch_size=32,
        class_mode='categorical'
    )
    return train_generator,validation_generator
    print("completed")


In [61]:
train_dir='/content/new data/train'
test_dir='/content/new data/test'
train_generator,validation_generator=Generators(train_dir,test_dir)

Found 4308 images belonging to 4 classes.
Found 1080 images belonging to 4 classes.


In [67]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense,Dropout
from tensorflow.keras.optimizers import Adam

def cnn_model():
    model = Sequential()

    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(128, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(256, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Flatten())
    model.add(Dense(512, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(256,activation='relu'))
    model.add(Dropout(0.1))
    model.add(Dense(128,activation='relu'))
    model.add(Dense(64,activation='relu'))
    model.add(Dense(4, activation='softmax'))
    return model

def modified_cnn_model():
    model = Sequential()

    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(128, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(256, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(512, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Flatten())
    model.add(Dense(512, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(128, activation='relu'))
    model.add(Dense(4, activation='softmax'))

    return model



In [63]:
model =cnn_model()
model.summary()
model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_16 (Conv2D)          (None, 254, 254, 32)      896       
                                                                 
 max_pooling2d_16 (MaxPooli  (None, 127, 127, 32)      0         
 ng2D)                                                           
                                                                 
 conv2d_17 (Conv2D)          (None, 125, 125, 64)      18496     
                                                                 
 max_pooling2d_17 (MaxPooli  (None, 62, 62, 64)        0         
 ng2D)                                                           
                                                                 
 conv2d_18 (Conv2D)          (None, 60, 60, 128)       73856     
                                                                 
 max_pooling2d_18 (MaxPooli  (None, 30, 30, 128)      

In [64]:
history = model.fit_generator(train_generator,
                              steps_per_epoch=len(train_generator),
                              epochs=10,
                              validation_data=validation_generator,
                              validation_steps=len(validation_generator))

Epoch 1/10


<ipython-input-64-63389289fdb4>:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  history = model.fit_generator(train_generator,


135/135 [==============================] - 19s 122ms/step - loss: 0.6836 - accuracy: 0.7105 - val_loss: 0.3608 - val_accuracy: 0.8500
Epoch 2/10
135/135 [==============================] - 18s 129ms/step - loss: 0.4511 - accuracy: 0.8071 - val_loss: 0.4301 - val_accuracy: 0.8380
Epoch 3/10
135/135 [==============================] - 17s 124ms/step - loss: 0.4147 - accuracy: 0.8192 - val_loss: 0.3014 - val_accuracy: 0.8694
Epoch 4/10
135/135 [==============================] - 17s 127ms/step - loss: 0.3300 - accuracy: 0.8482 - val_loss: 0.2891 - val_accuracy: 0.8676
Epoch 5/10
135/135 [==============================] - 16s 120ms/step - loss: 0.3011 - accuracy: 0.8735 - val_loss: 0.3498 - val_accuracy: 0.8546
Epoch 6/10
135/135 [==============================] - 16s 120ms/step - loss: 0.2554 - accuracy: 0.8974 - val_loss: 0.2798 - val_accuracy: 0.8796
Epoch 7/10
135/135 [==============================] - 16s 122ms/step - loss: 0.2116 - accuracy: 0.9157 - val_loss: 0.3372 - val_accuracy: 0.8

In [66]:
loss,accuracy=model.evaluate(validation_generator ,steps=validation_generator.samples//validation_generator.batch_size)
print(f"Test Loss: {loss:.4f}")
print(f"test accuracy:{accuracy:.4f}")

33/33 [==============================] - 3s 86ms/step - loss: 0.5759 - accuracy: 0.8456
Test Loss: 0.5759
test accuracy:0.8456


**MODIFIED CNN MODEL**

In [68]:
model2 = modified_cnn_model()
model2.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

# Print model summary
model2.summary()


Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_20 (Conv2D)          (None, 254, 254, 32)      896       
                                                                 
 max_pooling2d_20 (MaxPooli  (None, 127, 127, 32)      0         
 ng2D)                                                           
                                                                 
 conv2d_21 (Conv2D)          (None, 125, 125, 64)      18496     
                                                                 
 max_pooling2d_21 (MaxPooli  (None, 62, 62, 64)        0         
 ng2D)                                                           
                                                                 
 conv2d_22 (Conv2D)          (None, 60, 60, 128)       73856     
                                                                 
 max_pooling2d_22 (MaxPooli  (None, 30, 30, 128)      

In [70]:
history2 = model2.fit_generator(train_generator,
                              steps_per_epoch=len(train_generator),
                              epochs=10,
                              validation_data=validation_generator,
                              validation_steps=len(validation_generator))

Epoch 1/10


<ipython-input-70-d42dc0e23029>:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  history2 = model2.fit_generator(train_generator,


135/135 [==============================] - 17s 124ms/step - loss: 0.4031 - accuracy: 0.8357 - val_loss: 0.3173 - val_accuracy: 0.8380
Epoch 2/10
135/135 [==============================] - 17s 128ms/step - loss: 0.3523 - accuracy: 0.8554 - val_loss: 0.2922 - val_accuracy: 0.8824
Epoch 3/10
135/135 [==============================] - 17s 125ms/step - loss: 0.3368 - accuracy: 0.8656 - val_loss: 0.2501 - val_accuracy: 0.8963
Epoch 4/10
135/135 [==============================] - 19s 139ms/step - loss: 0.3187 - accuracy: 0.8770 - val_loss: 0.2835 - val_accuracy: 0.8824
Epoch 5/10
135/135 [==============================] - 17s 122ms/step - loss: 0.3032 - accuracy: 0.8802 - val_loss: 0.2501 - val_accuracy: 0.8935
Epoch 6/10
135/135 [==============================] - 17s 122ms/step - loss: 0.2690 - accuracy: 0.8867 - val_loss: 0.2683 - val_accuracy: 0.8843
Epoch 7/10
135/135 [==============================] - 17s 129ms/step - loss: 0.2552 - accuracy: 0.9039 - val_loss: 0.2899 - val_accuracy: 0.8

In [71]:
loss,accuracy=model2.evaluate(validation_generator ,steps=validation_generator.samples//validation_generator.batch_size)
print(f"Test Loss: {loss:.4f}")
print(f"test accuracy:{accuracy:.4f}")

33/33 [==============================] - 3s 93ms/step - loss: 0.3004 - accuracy: 0.8722
Test Loss: 0.3004
test accuracy:0.8722
